In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Налаштування управління шумом з Estimator

<Accordion>
<AccordionItem title="Версії пакетів">

Код на цій сторінці було розроблено з використанням таких вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Існує кілька способів управління шумом, зазвичай шляхом використання різних технік пом'якшення помилок та придушення помилок, щоб уникнути помилок до того, як вони виникнуть. Ці техніки зазвичай викликають накладні витрати на попередню обробку. Тому важливо досягти балансу між вдосконаленням результатів та забезпеченням завершення завдання в розумний час.

Estimator підтримує такі техніки управління шумом. Дивись [Техніки пом'якшення та придушення помилок](error-mitigation-and-suppression-techniques) для пояснення кожної. Дивись розділ [Налаштування помилок вручну](#advanced-error) для інструкцій з увімкнення цих технік.

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Рівень стійкості
`resilience_level` визначає, наскільки стійкою до помилок має бути система. Вищі рівні генерують більш точні результати ціною тривалішого часу обробки. Рівні стійкості можна використовувати для налаштування компромісу між вартістю та точністю при застосуванні управління шумом до запиту до примітива. Управління шумом зменшує помилки (зміщення) в результатах шляхом обробки виходів від колекції або ансамблю пов'язаних схем. Ступінь зменшення помилок залежить від застосованого методу. Рівень стійкості абстрагує детальний вибір методу управління шумом, щоб дозволити користувачам міркувати про компроміс вартість/точність, що підходить їхньому застосунку.

З огляду на це, кожен рівень відповідає методу або методам із зростаючим рівнем накладних витрат квантової вибірки, щоб дозволити тобі експериментувати з різними компромісами між часом та точністю. У наступній таблиці показано, які рівні та відповідні методи доступні для кожного з примітивів.

<span id="resilience-table"></span>

| Рівень стійкості | Опис | Техніка |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | Без пом'якшення | Відсутня |
| 1 [За замовчуванням] | Мінімальні витрати на пом'якшення: пом'якшення помилок зчитування | Вимірювальне twirling [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) |
| 2 | Середні витрати на пом'якшення. Зазвичай зменшує зміщення в оцінках, але не гарантовано бути нульовим зміщенням. | Рівень 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) та gate twirling

> **Info:** Рівні стійкості наразі перебувають у бета-версії, тому накладні витрати вибірки та якість рішення варіюватимуться від схеми до схеми. Нові функції, розширені опції та інструменти управління будуть випускатися поступово. Конкретні методи управління шумом не гарантовано застосовуватися на кожному рівні стійкості.

### Налаштування Estimator з рівнями стійкості
Можна використовувати рівні стійкості для вказання технік управління шумом або налаштовувати окремі техніки індивідуально, як описано в [Налаштування помилок вручну](#advanced-error).

> **Note:** Будь-які опції, які ти встановлюєш вручну на додаток до рівня стійкості, застосовуються поверх базового набору опцій, визначених рівнем стійкості. Тому, в принципі, можна встановити рівень стійкості 1, але потім вимкнути пом'якшення вимірювань, хоча це не рекомендується.
> 
> Наприклад, встановлення рівня стійкості 0 вимикає `zne_mitigation`, але `estimator.options.resilience.zne_mitigation = True` перевизначає це значення.

### Приклад
Наступний код вмикає ZNE, TREX та gate twirling, встановлюючи `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Налаштування управління шумом вручну
Можна вмикати та вимикати окремі методи управління шумом, використовуючи [опції Estimator](/guides/estimator-options).

> **Note:** Не всі опції працюють разом на всіх типах схем. Дивись [таблицю сумісності функцій](estimator-options#options-compatibility-table) для деталей.

### Приклад

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>